In [33]:
import pandas as pd

df = pd.read_csv('data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            248 non-null    str    
 1   selic           248 non-null    float64
 2   ipca            248 non-null    float64
 3   ibc_br          248 non-null    float64
 4   preco_gasolina  248 non-null    float64
dtypes: float64(4), str(1)
memory usage: 9.8 KB


In [19]:
df.head(5)

,Date,selic,ipca,ibc_br,preco_gasolina
0,2005-05-31,19.60,0.49,178129.5,1.814208
1,2005-06-30,19.75,-0.02,179509.8,1.775512
2,2005-07-31,19.75,0.25,181673.2,1.785572
3,2005-08-31,19.75,0.17,186712.8,1.793754
4,2005-09-30,19.62,0.35,184473.1,1.903378


In [34]:
import numpy as np

df['ipca_index'] = (1 + df['ipca']/100).cumprod()

df['log_ibc'] = np.log(df['ibc_br'])
df['log_gasolina'] = np.log(df['preco_gasolina'])
df['log_ipca'] = np.log(df['ipca_index'])

df_vecm = df[[
    'selic',
    'log_ipca',
    'log_ibc',
    'log_gasolina'
]].dropna()

In [35]:
df_vecm.head(5)

,selic,log_ipca,log_ibc,log_gasolina
0,19.60,0.004888,12.090266,0.595649
1,19.75,0.004688,12.097985,0.574089
2,19.75,0.007185,12.109965,0.579739
3,19.75,0.008883,12.137327,0.584311
4,19.62,0.012377,12.125259,0.643630


In [36]:
endog = df_vecm[['log_ipca', 'log_ibc', 'log_gasolina']]
exog = df_vecm[['selic']]

In [37]:
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron


def test_stationarity(series, col):
    series = series.dropna()
    
    # ADF
    adf = adfuller(series)
    
    # PP
    pp = PhillipsPerron(series)
    
    # KPSS
    kpss_test = kpss(series, regression='c', nlags='auto')
    
    return {
        "Variavel": col,
        "ADF_pvalue": adf[1],
        "PP_pvalue": pp.pvalue,
        "KPSS_pvalue": kpss_test[1]
    }


resultados = []
for col in endog.columns:
    res = test_stationarity(df[col], col)
    resultados.append(res)

frame_final = pd.DataFrame(resultados)
frame_final

C:\Users\Hertz Rafael\AppData\Local\Temp\ipykernel_24120\2436622687.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_test = kpss(series, regression='c', nlags='auto')
C:\Users\Hertz Rafael\AppData\Local\Temp\ipykernel_24120\2436622687.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_test = kpss(series, regression='c', nlags='auto')
C:\Users\Hertz Rafael\AppData\Local\Temp\ipykernel_24120\2436622687.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_test = kpss(series, regression='c', nlags='auto')


,Variavel,ADF_pvalue,PP_pvalue,KPSS_pvalue
0,log_ipca,0.948101,0.964358,0.01
1,log_ibc,0.568336,0.604843,0.01
2,log_gasolina,0.914397,0.932437,0.01


In [38]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

johansen = coint_johansen(endog, det_order=0, k_ar_diff=1)

print(johansen.lr1)   # estatística trace
print(johansen.cvt)   # valores críticos

[35.13631761  6.67105047  0.32215178]
[[27.0669 29.7961 35.4628]
 [13.4294 15.4943 19.9349]
 [ 2.7055  3.8415  6.6349]]


Em um primeiro resultado, foi possível identificar que, ao rodar johansen, foram encontrados 3 vetores de cointegração. sendo que, como a quantidade de vetores são iguais a quantidade de variaveis, isso significa que a série conjunta é estacionária, mesmo que individualmente sejam nao estacionarias.

In [39]:
from statsmodels.tsa.vector_ar.vecm import VECM

vecm = VECM(
    endog,
    exog=exog,
    k_ar_diff=1,
    coint_rank=1  # POSSUI NO MÁXIMO 1 VETOR (VALOR ENCONTRADO ATRAVES DE JOHANSEN)
)

vecm_fit = vecm.fit()

In [40]:
print(vecm_fit.summary())

Det. terms outside the coint. relation & lagged endog. parameters for equation log_ipca
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
exog1           -3.624e-05   4.64e-05     -0.781      0.435      -0.000    5.47e-05
L1.log_ipca         0.5103      0.058      8.867      0.000       0.397       0.623
L1.log_ibc         -0.0148      0.005     -3.289      0.001      -0.024      -0.006
L1.log_gasolina     0.0178      0.009      1.988      0.047       0.000       0.035
Det. terms outside the coint. relation & lagged endog. parameters for equation log_ibc
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
exog1              -0.0002      0.001     -0.279      0.780      -0.001       0.001
L1.log_ipca         0.2565      0.807      0.318      0.751      -1.3

log_ipca - 0.8410*log_ibc + 1.0828*log_gasolina = 0

### Isolando IPCA:
log(ipca)=0.8410⋅log(ibc)−1.0828⋅log(gasolina)


ou seja, 1% ↑ atividade → 0.84% ↑ inflação ; atividade foi significativa
no entanto, gasolina não foi significativo, ou seja, não há evidência estatística de relação de longo prazo entre preço da gasolina e inflação.



## Termo de correção de erro:
A inflação corrige desvios do equilíbrio de longo prazo, porém de forma lenta (0,0003).
IBC-br e gasolina foram não significativos.


## Curto prazo:
Inflação:

própria defasagem → significativa
ibc → impacto negativo
gasolina → impacto positivo (fraco)


IBC-Br
própria defasagem → significativa


Gasolina
própria defasagem → significativa
inflação → impacto positivo

## Resposta mais detalhada:

O modelo VECM estimado indicou a existência de uma relação de cointegração entre as variáveis, evidenciando um equilíbrio de longo prazo no sistema. A equação de cointegração mostrou que a atividade econômica possui relação positiva e estatisticamente significativa com a inflação, indicando que aumentos na atividade estão associados a elevações no nível de preços no longo prazo. Por outro lado, o preço da gasolina não apresentou significância estatística, não sendo possível afirmar sua influência no equilíbrio de longo prazo.

O termo de correção de erro foi significativo apenas na equação da inflação, com sinal negativo, indicando que essa variável é responsável pelo ajuste ao equilíbrio de longo prazo, ainda que de forma lenta. As demais variáveis não apresentaram mecanismos de ajuste significativos.

No curto prazo, observou-se que a inflação apresenta forte persistência, sendo influenciada por sua própria defasagem e pela atividade econômica. A variável preço da gasolina também apresentou comportamento inercial, além de influência da inflação.

A taxa Selic, incluída como variável exógena, não apresentou significância estatística, indicando ausência de efeito relevante no curto prazo sobre as variáveis analisadas.